# Where does cancer break p53?
## Part 1: Getting and exploring the mutation data

**Goal of this notebook:** download every TP53 mutation found in the TCGA PanCancer Atlas (about 10,000 tumours across ~32 cancer types), keep the missense mutations, and see where along the protein they fall.

**How to use it:** run each cell in order with **Shift + Enter**. Read the comments before running. After each section there's a short "check yourself" question: make sure you can answer it before moving on.

### Step 1: Load the tools
- `requests` talks to websites (we use it to ask cBioPortal for data)
- `pandas` handles tables (think Excel, but in code)
- `matplotlib` draws charts

In [ ]:
import requests
import pandas as pd
import matplotlib.pyplot as plt

print("Tools loaded.")

### Step 2: Find all the TCGA PanCancer Atlas studies
cBioPortal has a public **API**: a way for code to request data directly instead of clicking download buttons. First we ask for the list of all studies, then keep the ones from the PanCancer Atlas (their IDs end in `_tcga_pan_can_atlas_2018`).

In [ ]:
BASE = "https://www.cbioportal.org/api"
HEADERS = {"Accept": "application/json"}

studies = requests.get(f"{BASE}/studies", headers=HEADERS, timeout=60).json()
pancan = [s for s in studies if s["studyId"].endswith("_tcga_pan_can_atlas_2018")]

# A lookup table: study ID -> readable cancer name
study_names = {s["studyId"]: s["name"].replace(" (TCGA, PanCancer Atlas)", "") for s in pancan}

print(f"Found {len(pancan)} PanCancer Atlas studies:")
for sid, name in study_names.items():
    print(f"  {sid:40s} {name}")

### Step 3: Download every TP53 mutation from those studies
Each gene has an ID number. TP53's **Entrez Gene ID is 7157**. For each study we ask: *give me all mutations in gene 7157 across all samples.*

This takes a minute or two. We save the result to a CSV file so you never have to download it again.

In [ ]:
TP53_ID = 7157
all_rows = []

for sid in study_names:
    url = f"{BASE}/molecular-profiles/{sid}_mutations/mutations"
    params = {"sampleListId": f"{sid}_all", "entrezGeneId": TP53_ID, "projection": "SUMMARY"}
    try:
        r = requests.get(url, params=params, headers=HEADERS, timeout=120)
        r.raise_for_status()
        muts = r.json()
        for m in muts:
            all_rows.append({
                "study": sid,
                "cancer_type": study_names[sid],
                "sample": m.get("sampleId"),
                "protein_change": m.get("proteinChange"),
                "mutation_type": m.get("mutationType"),
                "residue": m.get("proteinPosStart"),
            })
        print(f"{sid:40s} {len(muts):5d} TP53 mutations")
    except Exception as e:
        print(f"{sid:40s} FAILED ({e})")

df = pd.DataFrame(all_rows)
df.to_csv("tp53_mutations_raw.csv", index=False)
print(f"\nTotal: {len(df)} mutations saved to tp53_mutations_raw.csv")

### Step 4: Look at what you downloaded
Always look at raw data before analyzing it. Check what columns you have, what kinds of mutations exist, and whether anything looks weird.

In [ ]:
display(df.head(10))
print("\nMutation types:")
print(df["mutation_type"].value_counts())

**Check yourself:** Which mutation type is most common? Does that match what you read about p53 being unusual among tumour suppressors?

### Step 5: Clean the data
We keep only **missense mutations** (one amino acid swapped for another), because those are what we can map onto the 3D structure. We also:
- drop rows with no residue number
- remove duplicates (the same mutation listed twice for the same sample)

In [ ]:
missense = df[df["mutation_type"] == "Missense_Mutation"].copy()
missense = missense.dropna(subset=["residue"])
missense["residue"] = missense["residue"].astype(int)
missense = missense.drop_duplicates(subset=["sample", "protein_change"])

# p53 is 393 amino acids long: anything outside that is an error
missense = missense[(missense["residue"] >= 1) & (missense["residue"] <= 393)]

missense.to_csv("tp53_missense_clean.csv", index=False)
print(f"{len(missense)} missense mutations in {missense['sample'].nunique()} tumour samples")

### Step 6: Find the hotspots
Count how many tumours have a mutation at each residue, then look at the top 10.

In [ ]:
counts = missense["residue"].value_counts().sort_index()

top10 = counts.sort_values(ascending=False).head(10)
print("Top 10 most frequently mutated residues:")
for res, n in top10.items():
    changes = missense[missense["residue"] == res]["protein_change"].value_counts()
    common = ", ".join(f"{c} ({k})" for c, k in changes.head(3).items())
    print(f"  Residue {res}: {n} tumours   most common: {common}")

in_dbd = counts[(counts.index >= 94) & (counts.index <= 292)].sum()
print(f"\n{in_dbd / counts.sum():.1%} of missense mutations fall in the DNA-binding domain (residues 94-292)")

**Check yourself:** Are R175, R248 and R273 in your top 10? What percentage of mutations fall in the DNA-binding domain, and why does that make sense given what p53 does?

### Step 7: Your first figure: a lollipop plot
Each vertical line is a residue; its height is how many tumours carry a mutation there. The shaded region is the DNA-binding domain.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.5))

ax.axvspan(94, 292, color="#f4d9c6", alpha=0.6, label="DNA-binding domain (94–292)")
ax.vlines(counts.index, 0, counts.values, color="#8a8a8a", linewidth=0.8)
ax.scatter(counts.index, counts.values, s=14, color="#c0392b", zorder=3)

# Label the six biggest hotspots
for res, n in counts.sort_values(ascending=False).head(6).items():
    wild_type = missense[missense["residue"] == res]["protein_change"].iloc[0][0]  # original amino acid letter
    ax.annotate(f"{wild_type}{res}", (res, n), textcoords="offset points", xytext=(0, 6), ha="center", fontsize=9, fontweight="bold")

ax.set_xlim(0, 394)
ax.set_xlabel("Residue position in p53")
ax.set_ylabel("Number of tumours with a missense mutation")
ax.set_title("TP53 missense mutations across ~32 cancer types (TCGA PanCancer Atlas)")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.savefig("fig1_lollipop.png", dpi=200)
plt.show()

### Step 8: Quick look by cancer type
Which cancers have the most TP53 missense mutations? You'll use this in Week 2 to compare hotspot patterns between cancers.

In [ ]:
by_cancer = missense.groupby("cancer_type")["sample"].nunique().sort_values(ascending=False)
print(by_cancer.head(12))

## Before moving to Part 2
Make sure you can explain, in your own words:
1. What the API call in Step 3 is doing, and what 7157 means
2. Why we kept only missense mutations
3. Why we removed duplicates
4. What your lollipop plot shows and why the spikes are where they are

Download the two CSV files and the figure (Files panel on the left), because Part 2 (the 3D structure) builds on them.